# 第13回 その差は、偶然でも出るか（解答例・教員用）


---


## 前回からの接続


前回は、論文の著者が公開したコードを動かし、論文の数を出し直しました。そのとき、Affinity Propagationのグループの数が、乱数（random number）のシードを決めないと、実行のたびに揺れることを見ました。

計算の結果は、乱数だけでなく、どの人やどの国がデータに入ったかでも揺れます。今日は、手元のデータで見えた差が、偶然でも出る程度のものか、どれくらいの幅で揺れるかを、くり返しの計算で確かめます。


---


## 今日の分析目標


あなたは職業訓練事業の評価担当です。職業訓練を受けた人は、受けなかった人より収入が多くなったと言えるか、多いならどれくらいかに答える。

訓練を受けるかどうかを、くじ引きで決めた実験の445人を使います。まず2つの群の収入の差を出し、次に、その差が偶然でも出るかを並べ替え検定（permutation test）で確かめます。最後に、差がどれくらいの幅で揺れるかを、ブートストラップ（bootstrap）で測ります。

もう1つのねらいは、乱数でデータを作り直す計算を何千回もくり返し、結果をリストにためて、分布として読めるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、職業訓練事業の評価担当です。仕事に就けずにいる人に、しばらく仕事の経験を積んでもらう訓練の事業があります。費用がかかる事業なので、続けるかどうかを決めるために、訓練で収入が増えたのかを報告しなければなりません。

訓練を受けた人の収入が多くても、それが訓練のおかげとは限りません。もともと働く意欲の高い人が、訓練を受けたのかもしれないからです。そこでこの事業では、参加を希望した人を、くじ引きで2つの群に分けました。


### 手元のデータ

1970年代のアメリカで行われた職業訓練の実験（National Supported Work Demonstration）のデータです。LaLonde（1986）が分析し、Dehejia & Wahba（1999）が、1974年の収入まで分かる445人に絞りました。Rajeev Dehejiaのページで公開されています。

1行が1人です。くじ引きで訓練を受けた185人と、受けなかった260人が並びます。答えの列は、訓練が終わったあとの1978年の年収（ドル）です。


### 最後に出す答え

訓練を受けた人と受けなかった人の年収の差、その差が偶然でも出るかを表すp値（p-value）、差がどれくらいの幅で揺れるかを表す区間です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 帰無仮説（null hypothesis）のもとでラベルを並べ替え、差の分布を作って、p値を出せる
- p値が小さいことと、差が大きいことを区別して説明できる
- 行を復元抽出（sampling with replacement）するブートストラップで、差の95%区間を出せる
- 平均の差と中央値の差で結論が変わる理由を、分布の形から説明できる
- 乱数を使う計算を`for`で何千回もくり返し、結果をリストにためて分布として読める。同じ手順を2回目に使うときは`def`にまとめられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### データの置き場所

- 役割：データの置き場所を決める
- 入力：なし
- 出力：なし


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、② のデータ解析（差を出し、くり返しの計算で確かめる）を中心に進み、最後に ③（報告に何と書くか）へ進みます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 訓練を受けた人の年収は、どれだけ多いか？（1節）
2. 偶然でも、この差は出るか？（2節）
3. 1万回並べ替えると、どうなるか？（3節）
4. 差は、どれくらいの幅で揺れるか？（4節）
5. 中央値で比べると、どうなるか？（5節）
6. 報告に、何と書くか？（6節）


---


## 1. 訓練を受けた人の年収は、どれだけ多いか？


準備ができました。まず、データを読み、訓練を受けた人と受けなかった人の年収を比べます。


### お手本：データを読む

- 役割：データを読み、使う列の名前を日本語にする
- 入力：データの場所
- 出力：表の大きさと、先頭の3人

`訓練`は、訓練を受けた人が1、受けなかった人が0です。`高卒資格なし`は、高校を卒業していない人が1です。`年収75`と`年収78`は、1975年（訓練の前）と1978年（訓練のあと）の年収（ドル）です。


In [ ]:
d = pd.read_csv(f'{DATA_DIR}/nsw.csv')
d = d.rename(columns={'treat': '訓練', 'age': '年齢',
                      'educ': '教育年数', 'nodegree': '高卒資格なし',
                      're75': '年収75', 're78': '年収78'})
cols = ['訓練', '年齢', '教育年数', '高卒資格なし',
        '年収75', '年収78']
print(d.shape)
d[cols].head(3)


445人です。訓練を受けた人と受けなかった人の数を数えます。


In [ ]:
d['訓練'].value_counts()


訓練を受けた人が185人、受けなかった人が260人です。


### お手本：群ごとの年収

- 役割：訓練の有無ごとに、1978年の年収の平均と中央値を出し、平均の差を表示する
- 入力：表`d`
- 出力：2行2列の表と、平均の差

`groupby('訓練')`で群に分け、`agg`で平均（`mean`）と中央値（`median`）を出します（第1回）。


In [ ]:
d.groupby('訓練')['年収78'].agg(['mean', 'median']).round(1)


In [ ]:
m = d.groupby('訓練')['年収78'].mean()
print(round(m[1] - m[0], 1))


訓練を受けた人の年収の平均は6349.1ドル、受けなかった人は4554.8ドルで、差は1794.3ドルです。中央値で比べても、訓練を受けた人のほうが多くなっています。


### お手本：年収の分布

- 役割：2つの群の年収を、ヒストグラムで重ねて描く
- 入力：表`d`
- 出力：ヒストグラム

`bins`は、棒の区切りです。ここでは0ドルから2000ドルきざみで区切ります。`alpha=0.5`で半透明にして、重なりが見えるようにします。


In [ ]:
bins = range(0, 62001, 2000)
for g, lab in [(0, '受けなかった'), (1, '受けた')]:
    plt.hist(d.loc[d['訓練'] == g, '年収78'], bins=bins,
             alpha=0.5, label=lab)
plt.xlabel('1978年の年収（ドル）')
plt.ylabel('人数')
plt.legend()
plt.show()


どちらの群も、0ドルのあたりに人が多く集まり、右へ長く裾を引いています。年収が0ドルの人、つまり1978年に収入の無かった人の割合を出します。


In [ ]:
d['無収入'] = d['年収78'] == 0
d.groupby('訓練')['無収入'].mean().round(3)


年収が0ドルの人は、訓練を受けた人では24.3%、受けなかった人では35.4%です。訓練を受けた人のほうが、収入の無い人が少なくなっています。


### 確かめる：くじ引きで、そろったか

くじ引きで分けたなら、訓練の前の性質は、2つの群でおおむねそろっているはずです。そろっていれば、1978年の年収の差は、訓練のせいだと考えられます。

- 役割：訓練の有無ごとに、年齢・教育年数・高卒資格なしの割合・1975年の年収の平均を出す
- 入力：表`d`
- 出力：2行4列の表


In [ ]:
before = ['年齢', '教育年数', '高卒資格なし', '年収75']
d.groupby('訓練')[before].mean().round(2)


年齢は25.05歳と25.82歳、教育年数は10.09年と10.35年で、ほぼそろっています。高卒資格なしの割合は0.83と0.71で、少しずれています。くじ引きでも、偶然のずれは少し出ます。

では、1978年の年収の差1794.3ドルも、偶然のずれでしょうか。


---


## 2. 偶然でも、この差は出るか？


訓練を受けた人の年収の平均は、1794.3ドル多いことが分かりました。けれども、くじ引きでも偶然のずれは出ます。この差は、偶然でも出る程度のものでしょうか。


### 帰無仮説：訓練に効果はない

まず、「訓練に効果はない」と仮に考えます。このように、確かめたいことを打ち消した仮説（hypothesis）を、帰無仮説と呼びます。

訓練に効果がないなら、1人1人の1978年の年収は、くじで訓練に当たっても外れても、同じだったはずです。2つの群の平均の差は、くじで誰がどちらの群に入ったかだけで決まります。

それなら、くじを引き直して（ラベルを並べ替えて）差を出せば、「効果がなくても、偶然でこれくらいの差は出る」という幅が分かります。


In [ ]:
# @title 図：ラベルを1回並べ替える
idx = [0, 1, 2, 185, 186, 187]
yy = d.loc[idx, '年収78'].round(0).astype(int).to_list()
tt = d.loc[idx, '訓練'].to_list()
ss = [0, 1, 0, 1, 0, 1]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, lab, title in [(axes[0], tt, '元のラベル'), (axes[1], ss, '並べ替えたラベル')]:
    ax.axis('off')
    ax.set_title(title)
    for r, (v, g) in enumerate(zip(yy, lab)):
        col = '#e76f51' if g == 1 else '#2a9d8f'
        ax.text(0.25, 0.85 - r * 0.15, f'{v:,}', ha='right', va='center')
        ax.text(0.55, 0.85 - r * 0.15, '受けた' if g == 1 else '受けなかった',
                ha='center', va='center', color='white',
                bbox=dict(boxstyle='round', facecolor=col))
plt.show()


左が元のラベル、右が並べ替えたラベルです。年収の列はそのままで、ラベル（受けた・受けなかった）だけを入れ替えます。並べ替えても、受けた人の数（ここでは3人）は変わりません。


### 新しい書き方：乱数で並べ替える

`np.random.default_rng(0)`は、乱数を作る道具（乱数生成器）を作ります。`0`は乱数のシードで、同じシードなら、何度実行しても同じ乱数が出ます。`rng.permutation(並び)`は、並びをばらばらに並べ替えます（第8回）。


In [ ]:
rng = np.random.default_rng(0)
print(rng.permutation([1, 1, 1, 0, 0, 0]))


1が3個、0が3個のまま、並びだけが変わりました。


### お手本：元のラベルでの差

- 役割：年収とラベルを配列にし、元のラベルでの平均の差`obs`を出す
- 入力：表`d`
- 出力：平均の差

`to_numpy()`は、列をnumpyの配列にします。`y[t == 1]`は、ラベルが1の人の年収だけを取り出します（真偽値で選ぶ。第1回）。


In [ ]:
y = d['年収78'].to_numpy()
t = d['訓練'].to_numpy()
obs = y[t == 1].mean() - y[t == 0].mean()
print(round(obs, 1))


1節と同じ1794.3ドルです。この`obs`（observedの略。観測した値）と、並べ替えたときの差を比べます。


### TODO①：1回並べ替えて、差を出す

シード0の乱数生成器を作り、ラベル`t`を並べ替えた`s`を作ってください。`s`で選んだ2つの群の、年収の平均の差を表示しましょう。`s`の中の1の数（`s.sum()`）も表示して、185人のままか確かめます。


In [ ]:
# TODO: ラベルを1回並べ替えて、平均の差を出す

# 解答例①：1回並べ替えて、差を出す
rng = np.random.default_rng(0)
s = rng.permutation(t)
print(s.sum())
print(round(y[s == 1].mean() - y[s == 0].mean(), 1))


並べ替えたあとも、1の数は185人のままです。平均の差は−559.9ドルで、元の1794.3ドルよりずっと小さく、符号も逆です。けれども、1回だけでは、たまたまかもしれません。くじを何回も引き直したら、差はどう散らばるでしょうか。


---


## 3. 1万回並べ替えると、どうなるか？


1回並べ替えると、差は小さくなりました。けれども、1回だけでは何とも言えません。くじの引き直しを1万回くり返し、差がどう散らばるかを見ます。


### おさらい：forとappend

空のリストを作り、`for`の中で`append`を使うと、くり返しの結果を1つずつためられます（第4回）。


In [ ]:
out = []
for k in range(3):
    out.append(k * 10)
print(out)


### お手本：1万回並べ替える

- 役割：ラベルを1万回並べ替え、そのたびの平均の差をリスト`perm`にためて、配列にする
- 入力：年収`y`とラベル`t`
- 出力：ためた数と、先頭の3つの差

乱数生成器は、セルの頭で作り直します。こうすると、TODO①を何回実行しても、このセルの結果は変わりません。`np.array`で、リストをnumpyの配列にします。


In [ ]:
rng = np.random.default_rng(0)
perm = []
for r in range(10000):
    s = rng.permutation(t)
    perm.append(y[s == 1].mean() - y[s == 0].mean())
perm = np.array(perm)
print(len(perm))
print(perm[:3].round(1))


1万個の差がたまりました。先頭の−559.9は、TODO①の差と同じです。同じシードから始めたからです。


### お手本：差の分布を描く

- 役割：1万個の差をヒストグラムにし、元のラベルでの差`obs`を赤い線で描く
- 入力：`perm`と`obs`
- 出力：ヒストグラムと、差の平均と標準偏差（standard deviation）

`plt.axvline`は、縦の線を引きます。


In [ ]:
plt.hist(perm, bins=50)
plt.axvline(obs, color='red')
plt.xlabel('並べ替えたときの、平均の差（ドル）')
plt.ylabel('回数')
plt.show()
print(perm.mean().round(1), perm.std().round(1))


並べ替えたときの差は、0ドルのまわりに山を作りました。平均は−3.6ドル、標準偏差は639.8ドルです。訓練に効果がなくても、偶然で±640ドルくらいの差は、よく出るということです。

赤い線の1794.3ドルは、山の右の裾の、ずっと外にあります。


### p値：偶然で、これ以上の差が出る割合

帰無仮説が正しいとき、観測した差と同じか、それより極端な差が出る割合を、p値と呼びます。ここでは、並べ替えた差の大きさ（絶対値）が、観測した差の大きさ以上になった割合です。

$$p = \frac{|\text{並べ替えた差}| \ge |\text{観測した差}| \text{となった回数}}{10000}$$

絶対値をとるのは、訓練で年収が減る向きの、大きな差も数えるためです。p値が小さいほど、「効果がないのに、偶然でこれだけの差が出た」とは考えにくくなります。よく使われる目安（有意水準）は0.05です。


### 新しい書き方：Trueの割合

真偽値の配列に`np.mean`を使うと、`True`を1、`False`を0として平均します。つまり、`True`の割合になります。`np.abs`は絶対値です。


In [ ]:
print(np.mean(np.array([True, False, True, True])))
print(np.abs(np.array([-3, 2])))


### TODO②：p値を出す

`perm`の絶対値が、`obs`の絶対値以上になった割合を、p値`p`として表示してください。そうなった回数（`True`の数）も表示しましょう。


In [ ]:
# TODO: p値を出す

# 解答例②：p値を出す
big = np.abs(perm) >= abs(obs)
print(big.sum())
p = np.mean(big)
print(p)


並べ替えた差が1794.3ドル以上に開いたのは、1万回のうち45回でした。p値は0.0045です。訓練に効果がないとしたら、こんなに大きな差が出るのは、1万回に45回しかありません。偶然では説明しにくい差です。

ただし、p値が分かるのは「偶然で出るか」までです。差がどれくらいなのかは、まだ分かりません。


---


## 4. 差は、どれくらいの幅で揺れるか？


差は、偶然では出にくいことが分かりました。では、訓練の効果は、どれくらいでしょうか。1794.3ドルは、たまたまこの445人が集まったときの値です。別の445人を集めて実験し直したら、差は変わるはずです。


### ブートストラップ：手元のデータから、集め直す

実験をやり直すことはできません。そこで、手元の185人を、訓練を受けた人全体の代わりとみなします。185人の中から、1人ずつくじで選んでは戻すことを185回くり返すと、もう1組の185人ができます。同じ人が2回以上選ばれることも、1回も選ばれないこともあります。このような選び方を、復元抽出と呼びます。

訓練を受けなかった260人からも、同じように260人を復元抽出します。そこで平均の差を出すことを何回もくり返し、差がどれくらい揺れるかを見ます。この方法を、ブートストラップと呼びます。


### 新しい書き方：復元抽出

`rng.choice(並び, 個数)`は、並びから、戻しながら（復元抽出で）指定した個数を選びます。


In [ ]:
rng = np.random.default_rng(0)
print(rng.choice([10, 20, 30, 40, 50], 5))


5個から5個を選びましたが、同じ値が2回以上出て、選ばれない値もあります。


In [ ]:
# @title 図：5人から、5人を復元抽出する
v = d.loc[[0, 1, 2, 3, 4], '年収78'].round(0).astype(int).to_list()
pick = np.random.default_rng(0).choice(5, 5)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, vals, title in [(axes[0], v, '手元の5人'),
                        (axes[1], [v[i] for i in pick], '復元抽出した5人')]:
    ax.axis('off')
    ax.set_title(title)
    for r, x in enumerate(vals):
        ax.text(0.5, 0.85 - r * 0.17, f'{x:,}', ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='#e8f1fb'))
plt.show()


左は、訓練を受けた人の先頭の5人の年収です。右は、そこから5人を復元抽出したものです。同じ人が何回か選ばれ、選ばれない人もいます。


### お手本：1回ぶん

- 役割：2つの群から、それぞれ同じ人数を復元抽出し、平均の差を出す
- 入力：年収`y`とラベル`t`
- 出力：平均の差

`y1`は訓練を受けた185人の年収、`y0`は受けなかった260人の年収です。


In [ ]:
y1 = y[t == 1]
y0 = y[t == 0]
rng = np.random.default_rng(0)
a = rng.choice(y1, len(y1))
b = rng.choice(y0, len(y0))
print(round(a.mean() - b.mean(), 1))


集め直した445人では、差は1794.3ドルではなくなりました。これを1万回くり返します。


### お手本：1万回くり返す

- 役割：1回ぶんを1万回くり返し、平均の差をリスト`boot`にためて、配列にする。ヒストグラムを描く
- 入力：`y1`と`y0`
- 出力：ヒストグラムと、差の平均と標準偏差


In [ ]:
rng = np.random.default_rng(0)
boot = []
for r in range(10000):
    a = rng.choice(y1, len(y1))
    b = rng.choice(y0, len(y0))
    boot.append(a.mean() - b.mean())
boot = np.array(boot)
plt.hist(boot, bins=50)
plt.axvline(obs, color='red')
plt.xlabel('集め直したときの、平均の差（ドル）')
plt.ylabel('回数')
plt.show()
print(boot.mean().round(1), boot.std().round(1))


今度は、山が観測した差1794.3ドルのまわりにできました。3節の並べ替えでは、山は0のまわりでした。並べ替えは「効果がないとしたら」の世界を、ブートストラップは「手元のデータが表す世界から、集め直したら」を作っています。

差の平均は1800.2ドル、標準偏差は670.0ドルです。集め直すたびに、差はこれくらい揺れます。


### 95%区間：真ん中の95%

1万個の差を小さい順に並べ、下から2.5%と上から2.5%を切り落とすと、真ん中の95%が残ります。この範囲を、95%ブートストラップ区間と呼びます。信頼区間（confidence interval）の作り方の1つ（パーセンタイル法）です。

`np.percentile(配列, q)`は、小さい順に並べて下からq%の位置の値（qパーセンタイル）を返します。`q`にリストを渡すと、まとめて返します。


In [ ]:
print(np.percentile([10, 20, 30, 40, 50], [0, 50, 100]))


### TODO③：95%区間を出す

`boot`の2.5パーセンタイル（percentile）と97.5パーセンタイルを表示してください。


In [ ]:
# TODO: 平均の差の95%区間を出す

# 解答例③：平均の差の95%区間
print(np.percentile(boot, [2.5, 97.5]).round(1))


95%区間は、550.9ドルから3162.4ドルです。区間は0を含まないので、差はプラスと言えそうです。けれども、区間は広く、訓練の効果は、年に約550ドルかもしれないし、3000ドルを超えるかもしれません。

p値が0.0045と小さくても、効果の大きさは、これだけの幅でしか分かっていません。


---


## 5. 中央値で比べると、どうなるか？


平均の差の幅が分かりました。けれども、1節で見たように、年収は右に長く裾を引いていました。平均は、少数の年収の高い人に引っぱられます。中央値で比べると、どうなるでしょうか。


### 確かめる：年収の高い人

- 役割：訓練を受けた人の年収を小さい順に並べ、いちばん高い3人を表示する。2つの群の中央値の差も出す
- 入力：`y1`と`y0`
- 出力：3人の年収と、中央値の差

`np.sort`は、小さい順に並べます。`[-3:]`は、うしろから3つです。


In [ ]:
print(np.sort(y1)[-3:].round(0))
print(round(np.median(y1) - np.median(y0), 1))


訓練を受けた人には、年収が3万ドルを超える人が3人います。中央値の差は1093.5ドルで、平均の差の1794.3ドルより小さくなりました。

中央値の差も、ブートストラップで揺れの幅を見たくなります。けれども、4節のくり返しのセルを写して、`mean`を`median`に書き換えるのは、2回目です。同じ手順を2回目に使いたくなったので、関数にまとめます。


### 新しい書き方：関数を、値として渡す

`np.median`のような関数は、変数に入れたり、別の関数に渡したりできます。`f = np.median`とすると、`f(...)`は`np.median(...)`と同じ働きをします。


In [ ]:
f = np.median
print(f([1, 2, 10]))
f = np.mean
print(f([1, 2, 10]))


### お手本：ブートストラップを関数にまとめる

- 役割：統計量を計算する関数`stat`を受け取り、4節と同じ手順で、2つの群の`stat`の差を1万回ためて返す
- 入力：`stat`（`np.mean`や`np.median`）
- 出力：1万個の差の配列

4節のセルの`a.mean()`を`stat(a)`に、`b.mean()`を`stat(b)`に書き換え、`def`の中に入れただけです。


In [ ]:
def boot_diff(stat):
    rng = np.random.default_rng(0)
    out = []
    for r in range(10000):
        a = rng.choice(y1, len(y1))
        b = rng.choice(y0, len(y0))
        out.append(stat(a) - stat(b))
    return np.array(out)


まず、`np.mean`を渡して、4節の`boot`と同じ結果になるかを確かめます。`np.allclose`は、2つの配列がほぼ同じかを返します（第11回）。


In [ ]:
bm = boot_diff(np.mean)
print(np.allclose(bm, boot))


`True`です。同じシードから同じ順に乱数を使うので、4節と同じ1万個の差になりました。関数にまとめても、手順は変わっていません。


### TODO④：中央値の差の95%区間

`boot_diff`に`np.median`を渡し、中央値の差の1万個を`bmed`にしてください。`bmed`の95%区間を表示しましょう。


In [ ]:
# TODO: 中央値の差の95%区間を出す

# 解答例④：中央値の差の95%区間
bmed = boot_diff(np.median)
print(np.percentile(bmed, [2.5, 97.5]).round(1))


中央値の差の95%区間は、−444.7ドルから3019.7ドルです。区間が0を含みます。中央値で比べると、訓練を受けた人の年収が多いとは、言い切れません。

平均の差には、収入の無い人が減ったこと（35.4%から24.3%）と、年収の高い人が何人かいることの、両方が効いています。中央値は、その両方に動かされにくいので、差がぼやけます。どちらの物差しで測ったかを、報告に書く必要があります。


---


## 6. 報告に、何と書くか？


平均と中央値で、言えることが違いました。最後に、2つの物差しの結果を1つの表にまとめて、報告に何と書くかを決めます。


### おさらい：辞書のリストを表にする

辞書を1行ぶんとしてリストにためると、`pd.DataFrame`で表になります。辞書のキーが、列の名前になります（第4回）。


In [ ]:
rows = [{'名前': 'A', '値': 1}, {'名前': 'B', '値': 2}]
pd.DataFrame(rows)


### お手本：2つの物差しを1つの表に

- 役割：平均と中央値のそれぞれで、差と95%区間を出し、1つの表にする
- 入力：`y1`・`y0`と関数`boot_diff`
- 出力：2行4列の表

`for name, stat in [...]:`は、リストの要素（名前と関数の組）を1つずつ取り出し、2つの変数に分けて入れます。


In [ ]:
rows = []
for name, stat in [('平均', np.mean), ('中央値', np.median)]:
    lo, hi = np.percentile(boot_diff(stat), [2.5, 97.5])
    rows.append({'物差し': name, '差': stat(y1) - stat(y0),
                 '下限': lo, '上限': hi})
pd.DataFrame(rows).round(1)


表にすると、平均の区間は0を含まず、中央値の区間は0を含むことが、並べて見えます。


### 答え

くじ引きで訓練の有無を決めた445人で、1978年の年収を比べました。

- 平均の差：訓練を受けた人のほうが、1794.3ドル多い。ラベルを1万回並べ替えると、これ以上の差が出たのは45回で、p値は0.0045。偶然では説明しにくい
- 差の幅：ブートストラップの95%区間は550.9ドルから3162.4ドル。効果はプラスと言えそうだが、大きさは幅をもってしか分からない
- 中央値の差：1093.5ドルで、95%区間は−444.7ドルから3019.7ドル。0を含むので、中央値では言い切れない

報告には、「訓練を受けた人の年収は、平均で約1800ドル多く、偶然では説明しにくい。ただし効果の大きさは年に約550ドルから3200ドルの幅があり、中央値で比べると差があるとは言い切れない」と書きます。平均の差には、収入の無い人が減ったことが効いています。


まだ言えないこと

- 誰に効いたか：445人をまとめて比べた。高校を卒業していない人など、一部の人に限ると、言えることが変わるかもしれない
- いつまで効くか：分かるのは1978年の1年だけ
- ほかの場所でも効くか：1970年代のアメリカで、参加を希望した人の結果である


---


## 目標に答えられたか


- 今日の目標は「職業訓練を受けた人は、受けなかった人より収入が多くなったと言えるか、多いならどれくらいか」でした。p値と区間を使って答えられますか？
- 1節で、くじ引きで分けたことが大事なのはなぜですか？
- 2節で、帰無仮説が正しいなら、ラベルを並べ替えてもよいのはなぜですか？
- 3節で、p値は何の割合ですか？p値が小さいと、何が言えますか？
- 4節で、ブートストラップの山が、並べ替えの山と違う場所にできたのはなぜですか？
- 5節で、平均と中央値で結論が変わったのはなぜですか？
- 6節で、報告に区間も書くのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：差が偶然でも出るかは、ラベルを並べ替えて確かめる（並べ替え検定）。差がどれくらいの幅で揺れるかは、行を復元抽出して確かめる（ブートストラップ）。どちらも、くり返した結果をためて分布として読む
- 帰無仮説（効果はない）のもとでは、ラベルを並べ替えても年収は変わらない。並べ替えた差の分布の外に、観測した差があれば、偶然では説明しにくい
- p値は「偶然で、これ以上の差が出る割合」で、効果の大きさではない。大きさは区間で示す
- 右に裾を引く分布では、平均と中央値で結論が変わることがある。どちらで測ったかを書く
- Python：乱数生成器`np.random.default_rng(シード)`、`rng.permutation`・`rng.choice`、くり返しの結果をリストにためて`np.array`、`np.mean(条件)`で割合、`np.percentile`。同じ手順を2回目に使うときは、関数を引数で渡す`def`にまとめる


---


## 次回へ


今日は、数の列（年収）を、2つの群で比べました。けれども、データには、年代やジャンルのように、区分で表される列もよくあります。

次回の問いは、行と列が両方とも区分の表を、どうやって1枚の地図にするか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

高校を卒業していない人（`高卒資格なし`が1の人）に限ると、訓練の効果はどうでしょうか。その人たちだけの表`dn`を作り、訓練を受けた人と受けなかった人の人数、年収の平均の差、並べ替え検定のp値（シード0で1万回）を表示してください。


In [ ]:
dn = d[d['高卒資格なし'] == 1]
yn = dn['年収78'].to_numpy()
tn = dn['訓練'].to_numpy()
print(len(dn))
print(dn['訓練'].value_counts())
obs_n = yn[tn == 1].mean() - yn[tn == 0].mean()
print(round(obs_n, 1))
rng = np.random.default_rng(0)
perm_n = []
for r in range(10000):
    s = rng.permutation(tn)
    perm_n.append(yn[s == 1].mean() - yn[s == 0].mean())
perm_n = np.array(perm_n)
print(np.mean(np.abs(perm_n) >= abs(obs_n)))


<details><summary>詰まったら</summary>

`dn = d[d['高卒資格なし'] == 1]`で絞り、`yn = dn['年収78'].to_numpy()`、`tn = dn['訓練'].to_numpy()`とします。
3節のお手本の`y`を`yn`に、`t`を`tn`に書き換えれば、並べ替え検定になります。

</details>


### 応用②（判断）

応用①の人たちで、平均の差の95%ブートストラップ区間（シード0で1万回）を出してください。区間の下限は何ドルですか。小数第1位まで答えてください。


In [ ]:
y1n = yn[tn == 1]
y0n = yn[tn == 0]
rng = np.random.default_rng(0)
boot_n = []
for r in range(10000):
    a = rng.choice(y1n, len(y1n))
    b = rng.choice(y0n, len(y0n))
    boot_n.append(a.mean() - b.mean())
ci = np.percentile(boot_n, [2.5, 97.5]).round(1)
print(ci)
print('答え:', ci[0])


<details><summary>詰まったら</summary>

4節のお手本の`y1`・`y0`を、`yn[tn == 1]`・`yn[tn == 0]`に書き換えます。
`np.percentile(…, [2.5, 97.5]).round(1)`の1つめが下限です。

</details>


### 応用③（解釈）

職業訓練事業の評価担当として、445人全体の結果と、高校を卒業していない人に限った結果を、上司に3行程度で報告してください。差の大きさ、偶然で出るか（p値）、差の幅（区間）、そこから言えることと言えないこと、を入れましょう。


（模範例）

くじ引きで訓練の有無を決めた445人では、訓練を受けた人の1978年の年収は平均で1794.3ドル多く、並べ替え検定のp値は0.0045で、偶然では説明しにくい差でした。ただし、95%区間は550.9ドルから3162.4ドルと幅が広く、効果の大きさははっきりしません。

高校を卒業していない348人に限ると、差は1154.0ドルに縮み、p値は0.0965で、偶然でも出る範囲です。95%区間も−245.0ドルから2675.9ドルで、0を含むので、この人たちに効いたとは言い切れません。

ただし、効かなかったとも言えません。人数を絞ると、差の揺れが大きくなるからです。この人たちに効くかを確かめるには、もっと多くの人で調べる必要があります。


---


## 発展（任意）


### Gap統計：構造のないデータと比べて、グループの数を決める

第11回では、2007年の142か国を、平均寿命と対数GDPで3つのグループに分けました。けれども、グループの数$K=3$は人が決めました。慣性（中心までの距離の二乗和）は、$K$を増やすほど必ず下がるので、慣性がいちばん小さい$K$を選ぶことはできません。

Gap統計（Tibshiraniほか, 2001）は、今日の並べ替え検定と同じ考え方を使います。グループの構造がまったく無いデータ（参照データ）を乱数で何組も作り、同じ$K$で分けたときの慣性と比べます。本物のデータの慣性が、参照データよりずっと小さければ、その$K$のグループには、偶然では出ない構造があると考えます。

$$\mathrm{Gap}(K) = \overline{\log W^{*}_K} - \log W_K$$

$W_K$は$K$個に分けたときの慣性、$\overline{\log W^{*}_K}$は参照データの$\log W_K$の平均です。参照データは、標準化した2列の最小値から最大値までの長方形の中に、一様な乱数で142個の点をまいて作ります。


### お手本：142か国を準備する

- 役割：第11回と同じように、2007年の142か国の平均寿命と対数GDPを標準化した`X`を作る
- 入力：gapminderのデータ
- 出力：`X`の大きさ


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
gm = pd.read_csv(f'{DATA_DIR}/gapminder.csv')
g7 = gm[gm['year'] == 2007].reset_index(drop=True)
g7['対数GDP'] = np.log10(g7['gdpPercap'])
X = StandardScaler().fit_transform(g7[['lifeExp', '対数GDP']])
print(X.shape)


### お手本：Kごとに、Gapを出す

- 役割：参照データを50組作り、$K=1$〜$8$のそれぞれで、Gapと、その標準誤差`se`を辞書にためる
- 入力：`X`
- 出力：Kごとの表

`rng.uniform(下限, 上限, 大きさ)`は、下限から上限までの一様な乱数を返します。参照データの組と$K$の、二重のくり返しです。標準誤差は、参照データの$\log W_K$の標準偏差に$\sqrt{1 + 1/50}$を掛けたものです。


In [ ]:
def logW(Z, k):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    return np.log(km.inertia_)

rng = np.random.default_rng(0)
lo, hi = X.min(axis=0), X.max(axis=0)
refs = [rng.uniform(lo, hi, X.shape) for b in range(50)]
gap, se = {}, {}
for k in range(1, 9):
    ref_logW = np.array([logW(R, k) for R in refs])
    gap[k] = ref_logW.mean() - logW(X, k)
    se[k] = ref_logW.std() * np.sqrt(1 + 1 / 50)
pd.DataFrame({'Gap': gap, '標準誤差': se}).round(3)


### 1標準誤差の規則

Gapがいちばん大きい$K$を選ぶと、少しの得のために、グループを細かくしすぎることがあります。そこで、Tibshiraniらは、次の規則を勧めています。

$$\mathrm{Gap}(K) \ge \mathrm{Gap}(K+1) - \mathrm{se}(K+1)$$

を満たす、いちばん小さい$K$を選びます。$K$を1つ増やしても、Gapが標準誤差の分より大きくは増えないなら、そこで止めるということです。


In [ ]:
for k in range(1, 8):
    if gap[k] >= gap[k + 1] - se[k + 1]:
        print('選ぶK:', k)
        break
ks = list(gap)
plt.errorbar(ks, [gap[k] for k in ks], yerr=[se[k] for k in ks],
             marker='o', capsize=4)
plt.xlabel('K')
plt.ylabel('Gap')
plt.show()


$K=1$（分けない）のGapはとても小さく、1標準誤差の規則では退けられます。142か国には、構造のない一様なデータでは出ない、まとまりがあるということです。

規則が選ぶのは$K=2$です。第11回の$K=3$は、規則では選ばれません。けれども、$K=2$と$K=3$のGapの差は、標準誤差より小さく、ほとんど違いません。また、Gapがいちばん大きいのは、$K=6$です。規則は、データから言える最小限の数を教えてくれますが、報告に使う数は、目的（支援の方針をいくつに分けたいか）とあわせて決めます。

試すなら、参照データの組を50から20に減らして、選ぶ$K$が変わるかを見てみましょう。
